# Phase 2D - Nodule detection on LUNA16
### External validation of the Phase 2S segmenter: sensitivity, FROC, CPM

**Why detection and not Dice.** LUNA16 annotates nodules as **centroid + diameter**
(`annotations.csv`; 1186 nodules >=3 mm at 3-of-4 radiologist consensus). There are no
per-voxel nodule masks, so Dice there would be scored against spheres we drew
ourselves. Centroid + diameter is, however, exactly the right annotation for
**detection** metrics, which is what LUNA16 was built for. Dice comes from MSD
Task06 in Phase 2S; this notebook covers what LUNA16 genuinely supports.

**Run on:** Colab, free T4. Budget ~45-60 min (download ~10-15 min, inference ~20 min).

---

### What is being measured

The Phase 2S U-Net, trained on MSD Task06 tumour masks, applied unchanged to LUNA16.
Predicted masks are split into 3D connected components; each component is a candidate
detection scored by its mean predicted probability. Detections are matched to ground
truth using the **official LUNA16 criterion**: a detection counts as a hit if its
centroid falls inside the annotated nodule's radius, and each nodule can be hit once.

This is **external validation with zero fine-tuning**: different dataset, different
scanner population, different annotation protocol, and a model trained on large NSCLC
tumours now being asked to find small nodules. Expect the sensitivity to be low. That
is a legitimate and informative result, and it is reported as-is.

### Three honesty constraints, fixed up front

1. **Subset 0 only** (89 of 888 scans). The full 10-subset benchmark is ~60 GB and
   10-fold cross-validation is out of reach on free Colab. Our CPM is therefore **not
   directly comparable to published LUNA16 leaderboard numbers**, and the paper must
   say "subset 0, single fold" every time the number appears.
2. **No training on LUNA16.** The model never sees LUNA16 labels. Nothing is tuned
   here, not even the threshold, which is inherited from Phase 2S.
3. **Domain mismatch is the point,** not a defect to hide. MSD tumours are large NSCLC
   masses; LUNA16 targets are >=3 mm nodules.

## 1. Environment

In [ ]:
import subprocess, sys
for pkg in ["SimpleITK", "opencv-python-headless", "tabulate"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import tensorflow as tf, keras
gpus = tf.config.list_physical_devices("GPU")
print("tensorflow:", tf.__version__, "| keras:", keras.__version__)
print("GPUs      :", gpus)
assert gpus, "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."

In [ ]:
import os, json, glob, shutil, time, zipfile
import numpy as np
import pandas as pd
import cv2
import SimpleITK as sitk
from scipy import ndimage
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- must match Phase 2S ----
IMG_SIZE     = 256
BASE_FILTERS = 32
HU_LEVEL, HU_WIDTH = -600, 1500
SEG_THRESHOLD = 0.5        # OVERWRITE with `selected_threshold` from Phase 2S results.json
# -----------------------------
MIN_COMPONENT_VOXELS = 8   # discard specks below this size; reported, not silent
FP_POINTS = [0.125, 0.25, 0.5, 1, 2, 4, 8]   # official LUNA16 FROC operating points

RESULTS_DIR = "/content/fyp_phase2d_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED, "config": {
    "img_size": IMG_SIZE, "seg_threshold": SEG_THRESHOLD,
    "min_component_voxels": MIN_COMPONENT_VOXELS,
    "hu_level": HU_LEVEL, "hu_width": HU_WIDTH, "subset": "subset0 only (89/888 scans)"}}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("results ->", RESULTS_DIR)

## 2. Load the Phase 2S segmenter

Needs `unet_msd_best.weights.h5` from the Phase 2S results zip. Easiest path is
`"upload"`: it is only ~30 MB.

**Set `SEG_THRESHOLD` above to the `selected_threshold` recorded in the Phase 2S
`results.json` before running this.** Re-tuning the threshold against LUNA16 labels
would turn an external validation into a fitted result.

In [ ]:
WEIGHTS_SOURCE = "upload"        # "upload" | "url" | "drive"
WEIGHTS_URL    = ""              # used when WEIGHTS_SOURCE == "url"
DRIVE_PATH     = "/content/drive/MyDrive/FYP/unet_msd_best.weights.h5"
W = "/content/unet_msd_best.weights.h5"

if WEIGHTS_SOURCE == "upload":
    if not os.path.exists(W):
        from google.colab import files
        up = files.upload()
        shutil.move(list(up)[0], W)
elif WEIGHTS_SOURCE == "url":
    subprocess.run(["wget", "-q", "-O", W, WEIGHTS_URL], check=True)
elif WEIGHTS_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(DRIVE_PATH, W)
print(f"{os.path.getsize(W)/1e6:.1f} MB")

In [ ]:
# Architecture copied verbatim from Phase 2S so the weights load by structure.
def conv_block(x, f):
    for _ in range(2):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    return x

def build_unet(size=IMG_SIZE, base=BASE_FILTERS):
    inp = layers.Input((size, size, 1))
    skips, x = [], inp
    for i in range(4):
        x = conv_block(x, base * 2 ** i)
        skips.append(x)
        x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, base * 16)
    for i in reversed(range(4)):
        x = layers.Conv2DTranspose(base * 2 ** i, 2, strides=2, padding="same")(x)
        x = layers.Concatenate()([x, skips[i]])
        x = conv_block(x, base * 2 ** i)
    out = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inp, out, name="unet")

model = build_unet()
model.load_weights(W)
print("params:", f"{model.count_params():,}")

## 3. Download LUNA16 subset 0

From the Zenodo mirror, which needs no grand-challenge.org account. `subset0.zip` is
6.8 GB (89 scans in MetaImage `.mhd` + `.raw` format); `annotations.csv` is 137 KB and
holds all 1186 nodules across all ten subsets, filtered here to the scans we have.

In [ ]:
Z = "https://zenodo.org/records/3723295/files"
ROOT = "/content/luna16"; os.makedirs(ROOT, exist_ok=True)
ANN = f"{ROOT}/annotations.csv"

if not os.path.exists(ANN):
    subprocess.run(["wget", "-q", "-O", ANN, f"{Z}/annotations.csv"], check=True)

if not os.path.isdir(f"{ROOT}/subset0"):
    zp = f"{ROOT}/subset0.zip"
    if not os.path.exists(zp):
        t0 = time.time()
        subprocess.run(["wget", "-q", "--show-progress", "-O", zp, f"{Z}/subset0.zip"], check=True)
        print(f"downloaded in {(time.time()-t0)/60:.1f} min")
    with zipfile.ZipFile(zp) as z:
        z.extractall(ROOT)
    os.remove(zp)          # reclaim 6.8 GB

mhd = sorted(glob.glob(f"{ROOT}/subset0/*.mhd"))
print("scans:", len(mhd))
ann = pd.read_csv(ANN)
uids = {os.path.splitext(os.path.basename(p))[0] for p in mhd}
ann = ann[ann["seriesuid"].isin(uids)].reset_index(drop=True)
print("nodules in subset0:", len(ann), "across", ann["seriesuid"].nunique(), "scans")
print(ann["diameter_mm"].describe().round(2).to_string())
RESULTS["data"] = {"n_scans": len(mhd), "n_nodules": int(len(ann)),
                   "scans_with_nodules": int(ann["seriesuid"].nunique())}
save_json()

### 3.1 Coordinate handling

The classic LUNA16 footgun. `annotations.csv` gives `coordX/Y/Z` in **world
millimetres**, not voxels. Converting needs the scan's own origin and spacing:

```
voxel = (world - origin) / spacing        # (x, y, z)
```

while `sitk.GetArrayFromImage` returns the array as **(z, y, x)**. Getting either of
these backwards silently produces plausible-looking garbage, so the sanity check below
verifies that every annotated centroid actually lands inside its volume before any
scoring happens.

In [ ]:
def load_scan(path):
    itk = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(itk).astype(np.float32)      # (z, y, x), Hounsfield
    origin = np.array(itk.GetOrigin(), np.float64)            # (x, y, z) mm
    spacing = np.array(itk.GetSpacing(), np.float64)          # (x, y, z) mm
    direction = np.array(itk.GetDirection(), np.float64).reshape(3, 3)
    return arr, origin, spacing, direction

def world_to_voxel(world_xyz, origin, spacing):
    return (np.asarray(world_xyz, np.float64) - origin) / spacing      # (x, y, z)

def voxel_to_world(vox_xyz, origin, spacing):
    return np.asarray(vox_xyz, np.float64) * spacing + origin

In [ ]:
inside, outside, nonident = 0, 0, 0
for p in tqdm(mhd[:10], desc="coord check"):
    uid = os.path.splitext(os.path.basename(p))[0]
    rows = ann[ann["seriesuid"] == uid]
    if rows.empty:
        continue
    arr, origin, spacing, direction = load_scan(p)
    if not np.allclose(direction, np.eye(3), atol=1e-6):
        nonident += 1
    Z_, Y_, X_ = arr.shape
    for _, r in rows.iterrows():
        x, y, z = world_to_voxel([r.coordX, r.coordY, r.coordZ], origin, spacing)
        ok = (0 <= x < X_) and (0 <= y < Y_) and (0 <= z < Z_)
        inside += ok; outside += (not ok)

print(f"\nannotated centroids inside their volume: {inside}   outside: {outside}")
print(f"scans with a non-identity direction matrix: {nonident} (of 10 checked)")
assert outside == 0, "world->voxel conversion is wrong; do not trust anything downstream"
print("coordinate conversion OK")
RESULTS["coord_check"] = {"inside": int(inside), "outside": int(outside),
                          "non_identity_direction_in_10": int(nonident)}
save_json()

## 4. Run the segmenter and turn masks into detections

Per scan: window and CLAHE each axial slice exactly as in Phase 2S, predict, threshold,
then take **3D connected components** of the binary volume. Each component becomes one
candidate detection, with its confidence set to the mean predicted probability inside
it. Confidence is what the FROC curve sweeps, so it has to be a real number rather than
a constant.

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def hu_to_uint8(sl):
    lo, hi = HU_LEVEL - HU_WIDTH / 2.0, HU_LEVEL + HU_WIDTH / 2.0
    return (np.clip((sl - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

def prep_slice(sl):
    u8 = _clahe.apply(hu_to_uint8(sl))
    if u8.shape != (IMG_SIZE, IMG_SIZE):
        u8 = cv2.resize(u8, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return ((u8.astype(np.float32) - 127.0) / 127.0)[..., None]

def detect(path):
    # -> list of {world (x,y,z) mm, confidence, voxels}
    arr, origin, spacing, _ = load_scan(path)
    Z_, Y_, X_ = arr.shape
    probs = np.zeros((Z_, Y_, X_), np.float32)
    for s in range(0, Z_, 32):
        zs = list(range(s, min(s + 32, Z_)))
        batch = np.stack([prep_slice(arr[z]) for z in zs])
        p = model.predict(batch, verbose=0)[..., 0].astype(np.float32)
        for k, z in enumerate(zs):
            probs[z] = cv2.resize(p[k], (X_, Y_), interpolation=cv2.INTER_LINEAR)
    del arr

    binm = probs >= SEG_THRESHOLD
    lab, n = ndimage.label(binm)
    dets, dropped = [], 0
    if n:
        # bincount, not ndimage.sum(np.ones_like(lab), ...) - the latter allocates a
        # second full int64 volume (~0.5 GB on a large scan) purely to count voxels.
        sizes = np.bincount(lab.ravel(), minlength=n + 1)[1:]
        keep = np.where(sizes >= MIN_COMPONENT_VOXELS)[0] + 1
        dropped = int(n - len(keep))
        if len(keep):
            idx = keep.tolist()
            cents = ndimage.center_of_mass(binm, lab, index=idx)     # (z, y, x)
            means = ndimage.mean(probs, lab, index=idx)
            for li, (cz, cy, cx), conf in zip(idx, cents, np.atleast_1d(means)):
                dets.append({"world": voxel_to_world([cx, cy, cz], origin, spacing),
                             "confidence": float(conf), "voxels": int(sizes[li - 1])})
    del probs, binm, lab
    return dets, dropped

In [ ]:
all_dets, total_dropped = {}, 0
t0 = time.time()
for p in tqdm(mhd, desc="scans"):
    uid = os.path.splitext(os.path.basename(p))[0]
    d, dr = detect(p)
    all_dets[uid] = d; total_dropped += dr

n_det = sum(len(v) for v in all_dets.values())
print(f"\ninference: {(time.time()-t0)/60:.1f} min")
print(f"candidate detections kept   : {n_det}  ({n_det/len(mhd):.1f} per scan)")
print(f"components dropped as < {MIN_COMPONENT_VOXELS} voxels: {total_dropped}")
RESULTS["detections"] = {"kept": int(n_det), "per_scan": n_det / len(mhd),
                         "dropped_below_min_size": int(total_dropped)}
save_json()

## 5. Match detections to ground truth

**Official LUNA16 criterion:** a detection is a true positive when its centre lies
within the annotated nodule's radius. Each ground-truth nodule can be matched at most
once; when several detections fall inside one nodule the highest-confidence one is
kept and the rest count as false positives. Every unmatched detection is a false
positive, and every unmatched nodule is a miss.

In [ ]:
records, gt_rows, matched_gt, gt_total = [], [], 0, 0
for p in mhd:
    uid = os.path.splitext(os.path.basename(p))[0]
    gts = ann[ann["seriesuid"] == uid]
    gt_total += len(gts)
    dets = sorted(all_dets[uid], key=lambda d: -d["confidence"])
    used = set()
    for d in dets:
        hit = None
        for gi, g in gts.iterrows():
            if gi in used:
                continue
            dist = np.linalg.norm(d["world"] - np.array([g.coordX, g.coordY, g.coordZ]))
            if dist <= g.diameter_mm / 2.0:
                hit = gi; break
        if hit is not None:
            used.add(hit); matched_gt += 1
        records.append({"uid": uid, "confidence": d["confidence"],
                        "voxels": d["voxels"], "tp": hit is not None})
    # Per-nodule status recorded from the SAME greedy matching, so the size breakdown
    # in 6.1 cannot disagree with the headline sensitivity.
    for gi, g in gts.iterrows():
        gt_rows.append({"uid": uid, "diameter_mm": g.diameter_mm, "detected": gi in used})

det_df = pd.DataFrame(records, columns=["uid", "confidence", "voxels", "tp"])
det_df["tp"] = det_df["tp"].astype(bool)
gt_df = pd.DataFrame(gt_rows, columns=["uid", "diameter_mm", "detected"])
print(f"ground-truth nodules      : {gt_total}")
print(f"nodules detected at least once: {matched_gt}  "
      f"(recall at max FP = {matched_gt/max(gt_total,1):.3f})")
print(f"total detections          : {len(det_df)}  "
      f"(TP {int(det_df['tp'].sum())} / FP {int((~det_df['tp']).sum())})")
det_df.to_csv(f"{RESULTS_DIR}/detections.csv", index=False)
RESULTS["matching"] = {"gt_nodules": int(gt_total), "nodules_hit": int(matched_gt),
                       "max_sensitivity": matched_gt / max(gt_total, 1),
                       "total_detections": int(len(det_df))}
save_json()

## 6. FROC and CPM

Detections are ranked by confidence and swept. At each threshold: sensitivity is the
fraction of ground-truth nodules hit, and the x-axis is average false positives **per
scan**. **CPM** is the mean sensitivity at 1/8, 1/4, 1/2, 1, 2, 4 and 8 FP/scan, which
is LUNA16's official summary statistic.

If the detector never reaches a given FP rate, that operating point scores its
sensitivity at the highest rate actually achieved, which is the conservative reading.

In [ ]:
n_scans = len(mhd)
d = det_df.sort_values("confidence", ascending=False).reset_index(drop=True)
tp_cum = d["tp"].cumsum().to_numpy()
fp_cum = (~d["tp"]).cumsum().to_numpy()
sens = tp_cum / max(gt_total, 1)
fppi = fp_cum / n_scans

froc = {}
for pt in FP_POINTS:
    ok = np.where(fppi <= pt)[0]
    froc[pt] = float(sens[ok[-1]]) if len(ok) else 0.0
CPM = float(np.mean(list(froc.values())))

print(f"{'FP/scan':>9}{'sensitivity':>14}")
for pt, s in froc.items():
    print(f"{pt:>9}{s:>14.4f}")
print(f"\nCPM (mean of the 7 points): {CPM:.4f}")
print(f"max FP/scan reached: {fppi[-1]:.2f}" if len(fppi) else "no detections")
RESULTS["froc"] = {str(k): v for k, v in froc.items()}
RESULTS["CPM"] = CPM
RESULTS["max_fppi_reached"] = float(fppi[-1]) if len(fppi) else 0.0
save_json()

In [ ]:
plt.figure(figsize=(6.5, 4.5))
plt.semilogx(np.maximum(fppi, 1e-3), sens, lw=2)
for pt, s in froc.items():
    plt.scatter([pt], [s], zorder=3)
plt.xlabel("average false positives per scan"); plt.ylabel("sensitivity")
plt.title(f"FROC - LUNA16 subset 0 (n={n_scans} scans, {gt_total} nodules)\nCPM = {CPM:.3f}")
plt.ylim(0, 1); plt.grid(alpha=.3, which="both")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_froc.png", dpi=150); plt.show()

### 6.1 Where the misses are

Sensitivity against nodule diameter. A model trained on large MSD tumours is expected
to miss small nodules, and this plot either confirms that or contradicts it. Either
way it is the substance of the discussion, not decoration.

In [ ]:
bins = [3, 5, 8, 12, 20, 100]
gt_df["size_bin"] = pd.cut(gt_df["diameter_mm"], bins)
by_size = gt_df.groupby("size_bin", observed=True)["detected"].agg(["mean", "count"]).round(3)
by_size.columns = ["sensitivity", "n_nodules"]
print(by_size.to_string())

plt.figure(figsize=(6.5, 4))
plt.bar(range(len(by_size)), by_size["sensitivity"])
plt.xticks(range(len(by_size)), [str(i) for i in by_size.index], rotation=20)
plt.ylabel("sensitivity"); plt.xlabel("nodule diameter (mm)")
plt.title("detection sensitivity by nodule size"); plt.ylim(0, 1); plt.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_sensitivity_by_size.png", dpi=150); plt.show()

gt_df.to_csv(f"{RESULTS_DIR}/gt_detection_status.csv", index=False)
RESULTS["sensitivity_by_size"] = {str(k): v for k, v in by_size.to_dict("index").items()}
save_json()

## 7. Bundle

In [ ]:
summary = pd.DataFrame([{
    "dataset": "LUNA16 subset0 (89/888 scans, single fold, no fine-tuning)",
    "gt_nodules": gt_total,
    "CPM": round(CPM, 4),
    "max_sensitivity": round(matched_gt / max(gt_total, 1), 4),
    "detections_per_scan": round(len(det_df) / n_scans, 2),
}])
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS_DIR}/detection_summary.csv", index=False)
with open(f"{RESULTS_DIR}/detection_summary.md", "w") as f:
    f.write(summary.to_markdown(index=False))

shutil.make_archive("/content/fyp_phase2d_results", "zip", RESULTS_DIR)
print("\n", sorted(os.listdir(RESULTS_DIR)))
try:
    from google.colab import files
    files.download("/content/fyp_phase2d_results.zip")
except Exception as e:
    print("download manually from the file browser:", e)

## 8. Output and interpretation

`fyp_phase2d_results.zip` contains `results.json`, the per-detection and per-nodule
CSVs, and the FROC and sensitivity-by-size figures.

Interpreting the outcome:

- **The coordinate assertion in §3.1 must pass.** Everything downstream is meaningless
  otherwise. It must not be removed to force the notebook through.
- **CPM near 0 with a large detection count** means the segmenter is firing on lung
  parenchyma rather than nodules. Verify `SEG_THRESHOLD` was set to the Phase 2S value.
- **CPM near 0 with almost no detections** is the opposite case: MSD-trained features do
  not respond to small nodules at all. Provided §3.1 passed, this is a genuine
  external-validation result rather than a defect, and is reported as such.
- **Disk exhaustion during extraction:** the archive is deleted after extraction, but
  ~14 GB is needed transiently. Restart the runtime and re-run.

Whatever the value, it is reported with "subset 0, single fold, no fine-tuning"
attached, since it is not comparable to full-benchmark leaderboard results.